# Introduction


This Notebook introduces Gemma 4:E2B (it).

The new model from Gemma series arrives in 4 parameter sizes:
* E2B
* E4B
* 26B
* 31B

We will test the multimodal and multilanguage capability of the most compact model E2B(it).

All models accept multimodal input (text, image, video) and output text.



# Upgrade transformers

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 83.6 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Define a pipeline

In [2]:
from transformers import pipeline
pipe = pipeline("any-to-any", model="google/gemma-4-e2b-it")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

# Test with image

Let's use here the example from HuggingFace.
We will analyze an image.

[](https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg)

<img src="https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg"></img>

In [3]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg",
            },
            {"type": "text", "text": "Do you have travel advice going to here?"},
        ],
    }
]
output = pipe(messages, max_new_tokens=100, return_full_text=False)
output[0]["generated_text"]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'The image you provided appears to show a **Buddhist temple or pagoda**, likely in **Southeast Asia** (such as Thailand, Myanmar, or Laos), given the architectural style of the tall, ornate spire.\n\n**To give you relevant travel advice, I need to know *where* this location is.**\n\nIf you can provide a name or location for this temple/area, I can give you specific advice on things like:\n\n* **Getting there:** Transportation, visa requirements.\n* **'

Let's beautify a bit the output.

In [4]:
from IPython.display import Markdown

display(Markdown(output[0]["generated_text"]))

The image you provided appears to show a **Buddhist temple or pagoda**, likely in **Southeast Asia** (such as Thailand, Myanmar, or Laos), given the architectural style of the tall, ornate spire.

**To give you relevant travel advice, I need to know *where* this location is.**

If you can provide a name or location for this temple/area, I can give you specific advice on things like:

* **Getting there:** Transportation, visa requirements.
* **

As we limited the number of tokens to 100, the output message is not fully displayed.

# Test with video


We continue now also with the video from HuggingFace example.

Let's first display the video.

In [5]:
from IPython.display import Video

Video("https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4")

In [6]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4",
            },
            {"type": "text", "text": "What is happening in this video?"},
        ],
    }
]

output = pipe(messages, load_audio_from_video=True)
output[0]["generated_text"]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user',
  'content': [{'type': 'video',
    'video': 'https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4'},
   {'type': 'text', 'text': 'What is happening in this video?'},
   {'type': 'audio'}]},
 {'role': 'assistant',
  'content': 'This video appears to show a large crowd of people gathered outdoors, seemingly watching a rocket launch or a large space event. The central focus of the image is a very tall rocket, which resembles a SpaceX Falcon 9, standing on a launch pad. There are also various other structures and people in the background. The sky is filled with clouds, suggesting it might be during sunrise, sunset, or an overcast day.'}]

Let's show the answer a bit beautified.

In [7]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

This video appears to show a large crowd of people gathered outdoors, seemingly watching a rocket launch or a large space event. The central focus of the image is a very tall rocket, which resembles a SpaceX Falcon 9, standing on a launch pad. There are also various other structures and people in the background. The sky is filled with clouds, suggesting it might be during sunrise, sunset, or an overcast day.

Not sure how accurate is the answer, since the rocket shows Arianne logo, but the answer is close enough.

# More tests


Let's try also with another images.


## A cow on the beach

<img src="https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"></img>

In [8]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"},
            {"type": "text", "text": "What you can see in this image?"}
        ]
    }
]

output = pipe(text=messages, max_new_tokens=300)
print(output[0]["generated_text"][-1]["content"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


This image features a **brown and white cow** standing on a **sandy beach** with the **ocean** in the background under a **bright, blue sky**.

Here's a breakdown of what can be seen:

*   **Subject:** A bovine animal (a cow), which has reddish-brown fur on its body and a prominent white patch on its face/head.
*   **Setting (Foreground/Ground):** The cow is standing on light-colored sand. There appears to be some wetness or shallow water near where the cow is standing, perhaps the edge of the surf or damp sand.
*   **Setting (Background):** There is a clear expanse of **ocean/sea** with turquoise or light blue water. In the distance, there are **landmasses or hills** visible along the horizon.
*   **Sky/Lighting:** The sky is bright blue with some white, wispy clouds, suggesting a sunny day.
*   **Overall Impression:** The scene evokes a warm, sunny, coastal, or beach environment.


In [9]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

This image features a **brown and white cow** standing on a **sandy beach** with the **ocean** in the background under a **bright, blue sky**.

Here's a breakdown of what can be seen:

*   **Subject:** A bovine animal (a cow), which has reddish-brown fur on its body and a prominent white patch on its face/head.
*   **Setting (Foreground/Ground):** The cow is standing on light-colored sand. There appears to be some wetness or shallow water near where the cow is standing, perhaps the edge of the surf or damp sand.
*   **Setting (Background):** There is a clear expanse of **ocean/sea** with turquoise or light blue water. In the distance, there are **landmasses or hills** visible along the horizon.
*   **Sky/Lighting:** The sky is bright blue with some white, wispy clouds, suggesting a sunny day.
*   **Overall Impression:** The scene evokes a warm, sunny, coastal, or beach environment.

# Small detail on a candy


<img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"></img>

In [10]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is represented on the candy?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [11]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Based on the image, the candies appear to be shaped like **bees**.

Actually, the small drawing on the candies are more like turtles.

Let's check now both the counting abilities of this compact model as well as German language knowledge.

In [12]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "Welche Farben haben die Bombons?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [13]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Die Bombons auf dem Bild haben die folgenden Farben:

* **Türkis/Grünblau** (zwei Stück)
* **Orange** (eins)
* **Grün** (eins)

The answer is perfect.

# Spanish culture and language


<img src="https://d1bv4heaa2n05k.cloudfront.net/user-images/1439905381602/shutterstock-78898486_destinationMain_1439905420657.jpeg"></img>

In [14]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://d1bv4heaa2n05k.cloudfront.net/user-images/1439905381602/shutterstock-78898486_destinationMain_1439905420657.jpeg"},
            {"type": "text", "text": "¿Qué ves en esta imagen?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [15]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

En la imagen se ve una escena de lo que parece ser una **exhibición o evento ecuestre/cultural**, con dos elementos principales:

1. **Una mujer vestida con un traje llamativo:**
    * Lleva un atuendo muy elaborado, con patrones dorados y negros.
    * Viste pantalones ajustados de color oscuro (posiblemente negro o morado oscuro).
    * Tiene detalles en colores brillantes, como rosa/fucsia en las mangas y en los zapatos.
    * Está sosteniendo una gran tela o capa de color **magenta o púrpura brillante** que se extiende a su alrededor.
    * Su postura es dinámica, como si estuviera bailando o realizando una demostración.

2. **Un toro negro:**
    * A la derecha de la mujer, hay un **toro grande y de color negro**, que parece estar en movimiento (corriendo o avanzando).
    * El toro tiene

The model is smart, but not that smart. It can interpret all the features and even small details in the image, but it is not recognizing a torrero scene.

# Japanese landmark

<img src="https://www.advantour.com/img/japan/tokyo/tokyo-tower.jpg"></img>

In [16]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://www.advantour.com/img/japan/tokyo/tokyo-tower.jpg"},
            {"type": "text", "text": "この画像には何が見えますか?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [17]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

この画像は、晴れた日の都市のパノラマビューを捉えています。

**主な特徴:**

* **東京のスカイライン:** 多くの高層ビルが立ち並ぶ都市の景色が広がっています。
* **東京タワー:** 画像の中央やや左寄りに、象徴的な赤い東京タワーがそびえ立っています。
* **空:** 非常に青く澄んだ快晴の空が広がり、明るい日差しの下で撮影されたことがわかります。
* **植生:** 都市の中にも緑が多く、特に前景や中景には木々（秋の紅葉を思わせる色合いも見られます）が見られ、自然と都市が共存している様子がうかがえます。
* **建築物:** 様々な高さとデザインの近代的なビル群が密集しています。

全体として、活気があり、美しいコントラストを持つ、日本の大都市（おそらく東京）の

The model was able to interpret correctly a landmark image from Tokyo, Tokyo Tower, with all details.

# French landmark

<img src="https://wmf.imgix.net/images/aa_fra_notre-dame_de_paris_0.jpg"></img>


In [18]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://wmf.imgix.net/images/aa_fra_notre-dame_de_paris_0.jpg"},
            {"type": "text", "text": "Que voyez-vous sur cette photo? Repondez succinte, svp."}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [19]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Cette photo montre la **Cathédrale Notre-Dame de Paris** avec son imposante façade en pierre, sous un ciel bleu parsemé de nuages. On aperçoit également une partie de la ville et de la foule au premier plan.

The answer is quite good, recognizing the Notre Dame cathedral.

# Conclusions

We checked the multimodal and multilanguage features of Gemma 3:2B (it) model.

The model can interpret both image and video.

We could verify that the model is able to interpret correctly a variety of images (describe the content of an inedite scene, perceive small details in a picture, correctly identify a landmark) and is also capable to process the text (and output answer) in multiple languages. 

Being a small size model (2B), it miss some of the contextual information (e.g. did not recognized a corrida) but in general it is impressive by the capacity to deal with multi-modal and multi-language data.

We used English, German, French, Spanish, and Japanese.